# Semi Supervised

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

OUT_DIR = Path("../outputs")
strong = pd.read_csv(OUT_DIR / "strong_labeled.csv")
weak   = pd.read_csv(OUT_DIR / "weak_labeled.csv")
LABELS = {"normal": 0, "cancer": 1}
strong["y"] = strong["y"].map(LABELS)
weak["y"]   = weak["y"].map(LABELS)
print("strong:", strong["y"].value_counts().to_dict(),
      "| weak:", weak["y"].value_counts().to_dict())

strong: {1: 50, 0: 50} | weak: {0: 892, 1: 514}


In [2]:
strong_train, strong_test = train_test_split(
    strong, test_size=0.30, stratify=strong["y"], random_state=42)

print("train:", len(strong_train), strong_train["y"].value_counts().to_dict())
print("test :", len(strong_test),  strong_test["y"].value_counts().to_dict())

overlap_train = set(strong_test["path"]) & set(strong_train["path"])
overlap_weak  = set(strong_test["path"]) & set(weak["path"])
print("Fuite test<->train fort :", len(overlap_train),
      "| test<->faible :", len(overlap_weak), "(doivent être 0)")

train: 70 {1: 35, 0: 35}
test : 30 {0: 15, 1: 15}
Fuite test<->train fort : 0 | test<->faible : 0 (doivent être 0)


In [3]:
IMG = 128
def load_img(path):
    im = Image.open(path).convert("L").resize((IMG, IMG))
    return np.asarray(im, dtype=np.float32) / 255.0

class MRIDataset(Dataset):
    def __init__(self, frame):
        self.paths = frame["path"].tolist(); self.y = frame["y"].tolist()
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = load_img(self.paths[i])[None, :, :]      # (1, 128, 128)
        return torch.from_numpy(x), torch.tensor(self.y[i], dtype=torch.long)

dl_weak  = DataLoader(MRIDataset(weak),         batch_size=64, shuffle=True)
dl_train = DataLoader(MRIDataset(strong_train), batch_size=16, shuffle=True)
dl_test  = DataLoader(MRIDataset(strong_test),  batch_size=32, shuffle=False)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(4))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3),
                                  nn.Linear(64*4*4, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x): return self.head(self.features(x))

xb, yb = next(iter(dl_test))
print("sortie :", SmallCNN().to(device)(xb.to(device)).shape, "(attendu [B, 2])")

sortie : torch.Size([30, 2]) (attendu [B, 2])


In [5]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

def train(model, loader, epochs, lr=1e-3):
    model.train(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        tot = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb)
            loss.backward(); opt.step(); tot += loss.item()*len(xb)
        print(f"  epoch {ep+1}/{epochs}  loss={tot/len(loader.dataset):.4f}")
    return model

@torch.no_grad()
def evaluate(model, loader, name=""):
    model.eval(); ys, ps = [], []
    for xb, yb in loader:
        ps.extend(model(xb.to(device)).argmax(1).cpu().numpy()); ys.extend(yb.numpy())
    ys, ps = np.array(ys), np.array(ps)
    rec = recall_score(ys, ps, pos_label=1, zero_division=0)
    print(f"[{name}] acc={accuracy_score(ys,ps):.3f} | "
          f"prec_cancer={precision_score(ys,ps,pos_label=1,zero_division=0):.3f} | "
          f"recall_cancer={rec:.3f} | F1_cancer={f1_score(ys,ps,pos_label=1,zero_division=0):.3f}")
    print("  matrice [[TN,FP],[FN,TP]] =\n", confusion_matrix(ys, ps))
    return rec

In [6]:
torch.manual_seed(0)
print("=== SUPERVISÉ (70 vraies labels) ===")
m_sup = SmallCNN().to(device)
train(m_sup, dl_train, epochs=15, lr=1e-3)
rec_sup = evaluate(m_sup, dl_test, "Supervisé")

=== SUPERVISÉ (70 vraies labels) ===
  epoch 1/15  loss=0.6979
  epoch 2/15  loss=0.6907
  epoch 3/15  loss=0.6801
  epoch 4/15  loss=0.6488
  epoch 5/15  loss=0.5929
  epoch 6/15  loss=0.5150
  epoch 7/15  loss=0.4577
  epoch 8/15  loss=0.4596
  epoch 9/15  loss=0.4836
  epoch 10/15  loss=0.4770
  epoch 11/15  loss=0.3910
  epoch 12/15  loss=0.4151
  epoch 13/15  loss=0.3957
  epoch 14/15  loss=0.3938
  epoch 15/15  loss=0.3683
[Supervisé] acc=0.700 | prec_cancer=0.667 | recall_cancer=0.800 | F1_cancer=0.727
  matrice [[TN,FP],[FN,TP]] =
 [[ 9  6]
 [ 3 12]]


In [7]:
torch.manual_seed(0)
print("=== SEMI-SUPERVISÉ — étape 1 : pré-entraînement sur 1406 labels faibles ===")
m_semi = SmallCNN().to(device)
train(m_semi, dl_weak, epochs=8, lr=1e-3)
print("\n-> perf après pré-entraînement faible SEUL (avant d'avoir vu une seule vraie label) :")
evaluate(m_semi, dl_test, "Semi (faible seul)")

print("\n=== étape 2 : affinage sur les 70 vraies labels ===")
train(m_semi, dl_train, epochs=15, lr=5e-4)   # lr réduit pour ne pas tout effacer
rec_semi = evaluate(m_semi, dl_test, "Semi (faible -> fort)")

=== SEMI-SUPERVISÉ — étape 1 : pré-entraînement sur 1406 labels faibles ===
  epoch 1/8  loss=0.6653
  epoch 2/8  loss=0.6570
  epoch 3/8  loss=0.6247
  epoch 4/8  loss=0.5807
  epoch 5/8  loss=0.5440
  epoch 6/8  loss=0.5183
  epoch 7/8  loss=0.4996
  epoch 8/8  loss=0.4763

-> perf après pré-entraînement faible SEUL (avant d'avoir vu une seule vraie label) :
[Semi (faible seul)] acc=0.667 | prec_cancer=0.857 | recall_cancer=0.400 | F1_cancer=0.545
  matrice [[TN,FP],[FN,TP]] =
 [[14  1]
 [ 9  6]]

=== étape 2 : affinage sur les 70 vraies labels ===
  epoch 1/15  loss=0.5962
  epoch 2/15  loss=0.5592
  epoch 3/15  loss=0.4786
  epoch 4/15  loss=0.4727
  epoch 5/15  loss=0.4566
  epoch 6/15  loss=0.4443
  epoch 7/15  loss=0.4107
  epoch 8/15  loss=0.4040
  epoch 9/15  loss=0.4368
  epoch 10/15  loss=0.4087
  epoch 11/15  loss=0.4428
  epoch 12/15  loss=0.3557
  epoch 13/15  loss=0.3702
  epoch 14/15  loss=0.3214
  epoch 15/15  loss=0.3424
[Semi (faible -> fort)] acc=0.833 | prec_cancer

In [8]:
print("\n===== COMPARAISON (test = 30 images jamais vues) =====")
print(f"Supervisé seul   : rappel cancer = {rec_sup:.3f}")
print(f"Semi-supervisé   : rappel cancer = {rec_semi:.3f}")
print(f"Gain semi - sup  : {rec_semi - rec_sup:+.3f}")


===== COMPARAISON (test = 30 images jamais vues) =====
Supervisé seul   : rappel cancer = 0.800
Semi-supervisé   : rappel cancer = 0.867
Gain semi - sup  : +0.067


In [9]:
from sklearn.metrics import recall_score, f1_score, accuracy_score

def quiet_train(model, loader, epochs, lr):
    model.train(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    return model

@torch.no_grad()
def metrics(model, loader):
    model.eval(); ys, ps = [], []
    for xb, yb in loader:
        ps.extend(model(xb.to(device)).argmax(1).cpu().numpy()); ys.extend(yb.numpy())
    ys, ps = np.array(ys), np.array(ps)
    return (recall_score(ys, ps, pos_label=1, zero_division=0),
            f1_score(ys, ps, pos_label=1, zero_division=0),
            accuracy_score(ys, ps))

res = {"supervisé": [], "semi": []}
for seed in range(5):
    torch.manual_seed(seed)
    ms = SmallCNN().to(device); quiet_train(ms, dl_train, 15, 1e-3)
    res["supervisé"].append(metrics(ms, dl_test))
    mm = SmallCNN().to(device)
    quiet_train(mm, dl_weak, 8, 1e-3); quiet_train(mm, dl_train, 15, 5e-4)
    res["semi"].append(metrics(mm, dl_test))
    print(f"seed {seed} ok")

for k, v in res.items():
    a = np.array(v)
    print(f"\n{k:10s} | rappel={a[:,0].mean():.3f}±{a[:,0].std():.3f} "
          f"| F1={a[:,1].mean():.3f}±{a[:,1].std():.3f} "
          f"| acc={a[:,2].mean():.3f}±{a[:,2].std():.3f}")

seed 0 ok
seed 1 ok
seed 2 ok
seed 3 ok
seed 4 ok

supervisé  | rappel=0.760±0.053 | F1=0.712±0.023 | acc=0.693±0.013

semi       | rappel=0.720±0.065 | F1=0.734±0.047 | acc=0.740±0.039


In [10]:
from sklearn.semi_supervised import LabelSpreading
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import recall_score, f1_score, accuracy_score

inv = pd.read_csv(OUT_DIR / "inventory.csv")
path2idx = {p: i for i, p in enumerate(inv["path"])}
X = np.load(OUT_DIR / "features.npy")
Xp = PCA(n_components=50, random_state=0).fit_transform(StandardScaler().fit_transform(X))

sl = pd.read_csv(OUT_DIR / "strong_labeled.csv")
sl["y"] = sl["y"].map({"normal": 0, "cancer": 1})

# même découpe que les CNN (random_state=42) -> on évalue sur le MÊME test
s_tr, s_te = train_test_split(sl, test_size=0.30, stratify=sl["y"], random_state=42)
y = np.full(len(inv), -1)                       # -1 = non étiqueté
for p, yy in zip(s_tr["path"], s_tr["y"]):
    y[path2idx[p]] = yy                          # seuls les 70 labels d'entraînement sont visibles
test_idx = [path2idx[p] for p in s_te["path"]]
y_true = s_te["y"].to_numpy()

lp = LabelSpreading(kernel="knn", n_neighbors=12, alpha=0.2, max_iter=60).fit(Xp, y)
pred = lp.transduction_[test_idx]
print("LABEL PROPAGATION — même test (30 IRM) que les CNN")
print(f"  rappel cancer = {recall_score(y_true, pred, pos_label=1, zero_division=0):.3f}"
      f" | F1 = {f1_score(y_true, pred, pos_label=1, zero_division=0):.3f}"
      f" | acc = {accuracy_score(y_true, pred):.3f}")

LABEL PROPAGATION — même test (30 IRM) que les CNN
  rappel cancer = 0.733 | F1 = 0.846 | acc = 0.867


In [11]:
true_all = sl["y"].to_numpy()
idx_all  = np.array([path2idx[p] for p in sl["path"]])
res = []
for seed in range(5):
    tr, te = next(StratifiedShuffleSplit(1, test_size=0.30, random_state=seed).split(idx_all, true_all))
    y = np.full(len(inv), -1); y[idx_all[tr]] = true_all[tr]
    lp = LabelSpreading(kernel="knn", n_neighbors=12, alpha=0.2, max_iter=60).fit(Xp, y)
    pred = lp.transduction_[idx_all[te]]; yt = true_all[te]
    res.append((recall_score(yt, pred, pos_label=1, zero_division=0),
                f1_score(yt, pred, pos_label=1, zero_division=0),
                accuracy_score(yt, pred)))
a = np.array(res)
print(f"LP sur 5 découpages | rappel={a[:,0].mean():.3f}±{a[:,0].std():.3f}"
      f" | F1={a[:,1].mean():.3f}±{a[:,1].std():.3f} | acc={a[:,2].mean():.3f}±{a[:,2].std():.3f}")

LP sur 5 découpages | rappel=0.867±0.042 | F1=0.909±0.029 | acc=0.913±0.027
